# ARLE DSpark Train on Colab

Train the DSpark Markov head online using the hybrid PG + supervised L1 loss.

**Requirements:**
- GPU runtime (T4 / A100 / L4)
- CUDA 12.x
- HuggingFace token (set `HF_TOKEN` below) for gated/private models

In [ ]:
# --- Config ---\nMODEL_ID = \"Qwen/Qwen3.5-0.8B\"          # main model (HF id)\nDRAFT_MODEL_ID = \"Qwen/Qwen3.5-0.8B-DFlash\"  # DSpark/DFlash draft checkpoint\nHF_TOKEN = \"\"  # @param {type:\"string\"} paste your HF token here\n\nimport os\nif HF_TOKEN:\n    os.environ[\"HF_TOKEN\"] = HF_TOKEN\n    os.environ[\"HUGGING_FACE_HUB_TOKEN\"] = HF_TOKEN

In [ ]:
# --- Check GPU ---
!nvidia-smi || echo "NO GPU DETECTED - change runtime to GPU"

In [ ]:
# --- Install Rust ---
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]

In [ ]:
# --- Clone ARLE ---
!git clone https://github.com/cklxx/arle.git 2>/dev/null || (cd arle && git pull)
%cd arle

In [ ]:
# --- Try to fetch prebuilt kernels (avoids TileLang install) ---\n# NOTE: Prebuilt kernels target sm_80/86/89/90 (A100/L4/H100/H20).\n# On T4 (sm_75) this will fail validation and fall back to TileLang regen.\nimport subprocess, os, json, urllib.request\n\ndef fetch_prebuilt_kernels():\n    \"\"\"Download the latest kernel bundle from GitHub releases into generated/.\"\"\"\n    repo = \"cklxx/arle\"\n    tag = \"kernel-artifacts\"\n    api = f\"https://api.github.com/repos/{repo}/releases/tags/{tag}\"\n    try:\n        with urllib.request.urlopen(api) as r:\n            release = json.load(r)\n    except Exception as e:\n        print(f\"Could not fetch release info: {e}\")\n        return False\n    # Find the t1 kernel bundle\n    asset = None\n    for a in release.get(\"assets\", []):\n        name = a[\"name\"]\n        if name.startswith(\"arle-kernels-t1-\") and name.endswith(\".tar.gz\"):\n            asset = a\n            break\n    if not asset:\n        print(\"No prebuilt kernel bundle found in releases.\")\n        return False\n    url = asset[\"browser_download_url\"]\n    print(f\"Downloading {asset['name']}...\")\n    !curl -L -o /tmp/kernels.tar.gz \"{url}\"\n    !rm -rf crates/cuda-kernels/generated\n    !mkdir -p crates/cuda-kernels/generated\n    !tar -xzf /tmp/kernels.tar.gz -C crates/cuda-kernels/generated\n    print(\"Prebuilt kernels extracted.\")\n    return True\n\nhave_kernels = fetch_prebuilt_kernels()\nprint(f\"Prebuilt kernels available: {have_kernels}\")

In [ ]:
# --- Install TileLang (only if no prebuilt kernels) ---
import os
if not os.path.exists('crates/cuda-kernels/generated') or not os.listdir('crates/cuda-kernels/generated'):
    print("Installing TileLang for kernel codegen...")
    !pip install -q -r requirements-build.txt
    # Set INFER_TILELANG_PYTHON to the current python
    import sys
    os.environ["INFER_TILELANG_PYTHON"] = sys.executable
    print(f"TileLang python: {sys.executable}")
else:
    print("Using prebuilt kernels, skipping TileLang install.")

In [ ]:
# --- Build arle ---
import os
os.environ["RUSTFLAGS"] = "-C target-cpu=native"
os.environ["ARLE_CUDA_KERNEL_CACHE"] = "0"  # use generated/ or regen
!cargo build --release --features cuda -p arle --bin arle 2>&1 | tail -20

In [ ]:
# --- Install huggingface_hub and download models ---\n!pip install -q huggingface_hub\n\nimport os\nfrom huggingface_hub import snapshot_download\n\nprint(f\"Downloading {MODEL_ID}...\")\nlocal_model = snapshot_download(MODEL_ID, token=HF_TOKEN)\nprint(f\"  -> {local_model}\")\n\nprint(f\"Downloading {DRAFT_MODEL_ID}...\")\nlocal_draft = snapshot_download(DRAFT_MODEL_ID, token=HF_TOKEN)\nprint(f\"  -> {local_draft}\")\n\n# Use local paths for serve\nos.environ[\"ARLE_MODEL_LOCAL\"] = local_model\nos.environ[\"ARLE_DRAFT_LOCAL\"] = local_draft\nprint(\"Models ready.\")

In [ ]:
# --- Start serve with DSpark train sidecar (background) ---\n# The sidecar spawns automatically when --spec-type dspark is used.\n# It drains the experience buffer populated by inference requests and\n# runs hybrid PG + L1 updates on the Markov head.\nimport subprocess, os, signal, time\n\ncmd = [\n    \"./target/release/arle\", \"serve\",\n    \"--model-path\", os.environ[\"ARLE_MODEL_LOCAL\"],\n    \"--spec-type\", \"dspark\",\n    \"--mtp-draft-model\", os.environ[\"ARLE_DRAFT_LOCAL\"],\n    \"--port\", \"8080\",\n]\n\nenv = os.environ.copy()\nlog_file = open(\"/tmp/arle.log\", \"w\")\nproc = subprocess.Popen(cmd, env=env, stdout=log_file, stderr=subprocess.STDOUT)\nprint(f\"Started arle serve (pid {proc.pid})\")\nprint(\"Waiting for server to be ready...\")\n\n# Wait for the server to come up\nfor i in range(60):\n    if proc.poll() is not None:\n        print(\"Server exited early! Check /tmp/arle.log\")\n        break\n    try:\n        import urllib.request\n        urllib.request.urlopen(\"http://127.0.0.1:8080/health\", timeout=2)\n        print(\"Server is ready!\")\n        break\n    except:\n        time.sleep(2)\nelse:\n    print(\"Server did not start within 120s\")

In [ ]:
# --- Check DSpark train sidecar started ---
!grep -i "dspark_train\|DSpark train" /tmp/arle.log | tail -10

In [ ]:
# --- Send inference requests to populate the experience buffer ---
# Each request generates DSpark draft steps; accepted/rejected tokens feed
# the trainer. Send several requests to accumulate experience.
import urllib.request, json

def chat(messages, max_tokens=128):
    payload = json.dumps({
        "model": MODEL_ID,
        "messages": messages,
        "max_tokens": max_tokens,
    }).encode()
    req = urllib.request.Request(
        "http://127.0.0.1:8080/v1/chat/completions",
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=60) as r:
        return json.load(r)

prompts = [
    "Explain quantum computing in simple terms.",
    "Write a haiku about machine learning.",
    "What are the benefits of Rust over C++?",
    "Describe the water cycle.",
    "Give 3 tips for better sleep.",
]

for i, p in enumerate(prompts):
    try:
        resp = chat([{"role": "user", "content": p}])
        text = resp["choices"][0]["message"]["content"][:80]
        print(f"[{i+1}/{len(prompts)}] {p[:40]}... -> {text}...")
    except Exception as e:
        print(f"[{i+1}] error: {e}")

In [ ]:
# --- Monitor training loss ---
# The trainer logs loss every step. Watch for decreasing values.
!grep "dspark_train:" /tmp/arle.log | tail -20

In [ ]:
# --- Live loss monitor (run this cell to stream new loss lines) ---
import subprocess, time
proc2 = subprocess.Popen(["tail", "-f", "/tmp/arle.log"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
try:
    for line in proc2.stdout:
        line = line.decode().strip()
        if "dspark_train:" in line:
            print(line, flush=True)
except KeyboardInterrupt:
    proc2.kill()
    print("Stopped monitoring.")

In [ ]:
# --- Stop the server ---
proc.kill()
proc.wait()
print("Server stopped.")
!grep "dspark_train:" /tmp/arle.log | tail -5